In [1]:
import gym  # importa o gym, framework de RL c os ambientes ja prontos
import random
random.seed(1234)  # fixa a seed pra dar sempre o msm resultado toda vez q roda

streets = gym.make("Taxi-v3").env  # cria o ambiente do taxi (mapa 5x5 c passageiro e destino)
streets.render()  # desenha o mapa no terminal p vc ver o estado atual

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+



In [2]:
# encode monta o num do estado a partir de (linha taxi, coluna taxi, ponto passageiro, ponto destino)
initial_state = streets.encode(2,3,2,0)
streets.s = initial_state  # forca o ambiente a comecar direto nesse estado
streets.render()

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+



In [3]:
# streets.P guarda, p cada estado, as transicoes possiveis por acao
# formato: {acao: [(probabilidade, prox_estado, reward, terminou)]}
streets.P[initial_state]

{0: [(1.0, 368, -1, False)],
 1: [(1.0, 168, -1, False)],
 2: [(1.0, 288, -1, False)],
 3: [(1.0, 248, -1, False)],
 4: [(1.0, 268, -10, False)],
 5: [(1.0, 268, -10, False)]}

In [4]:
import numpy as np

# tabela Q: uma linha p cada estado possivel, uma coluna p cada acao possivel
# comeca tudo zerado, o agente vai aprender os valores treinando
q_table = np.zeros([streets.observation_space.n, streets.action_space.n])

learning_rate = 0.1      # o qto o agente da peso pra info nova aprendida
discount_factor = 0.6    # o qto o agente valoriza recompensa futura
exploration = 0.1        # chance de fazer acao aleatoria (explorar) em vez da melhor conhecida
epochs = 10000           # qtd de "corridas" de treino

for taxi_run in range(epochs):
    state = streets.reset()  # reinicia o ambiente num estado aleatorio
    done = False
    
    while not done:  # roda ate a corrida acabar (passageiro entregue ou deu erro)
        random_value = random.uniform(0, 1)
        if (random_value < exploration):
            action = streets.action_space.sample() # Explore a random action -> acao aleatoria (exploracao)
        else:
            action = np.argmax(q_table[state]) # Use the action with the highest q-value -> melhor acao conhecida
            
        next_state, reward, done, info = streets.step(action)  # executa a acao no ambiente
        
        prev_q = q_table[state, action]
        next_max_q = np.max(q_table[next_state])
        # formula do Q-learning: mistura o valor antigo c a recompensa + estimativa futura
        new_q = (1 - learning_rate) * prev_q + learning_rate * (reward + discount_factor * next_max_q)
        q_table[state, action] = new_q
        
        state = next_state  # avanca p prox estado

In [5]:
# valores Q aprendidos p o estado inicial, um valor p cada uma das 6 acoes possiveis
# (qto maior o valor, melhor a acao segundo o q o agente aprendeu)
q_table[initial_state]

array([-2.41335114, -2.41537902, -2.41972719, -2.3639511 , -7.31086192,
       -9.57536622])

In [ ]:
from IPython.display import clear_output
from time import sleep

for tripnum in range(1, 11):  # simula 10 corridas usando a tabela Q ja treinada
    state = streets.reset()
    
    done = False
    
    while not done:
        action = np.argmax(q_table[state])  # sempre pega a melhor acao (sem exploracao aqui)
        next_state, reward, done, info = streets.step(action)
        clear_output(wait=True)  # limpa a tela p dar aquele efeito de "animacao"
        print("Trip number " + str(tripnum))
        print(streets.render(mode='ansi'))
        sleep(.5)  # pausa curta p da p acompanhar
        state = next_state
        
    sleep(2)  # pausa maior entre uma corrida e outra

Trip number 9
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (West)

